# Question 3 - Sentiment Classification Pipeline

This notebook builds a positive/negative sentiment classifier with Bag-of-Words (Count Vectorizer) and Logistic Regression. It is intended for the Lief technical test and can run from top to bottom.

## Workflow

1. Download and load the dataset. (ดาวน์โหลดและโหลดชุดข้อมูล)
2. Lowercase text, remove punctuation and the required stopwords, then tokenize. (แปลงตัวอักษรเป็นพิมพ์เล็ก ลบเครื่องหมายวรรคตอนและ stopwords ที่กำหนด แล้วตัดคำ)
3. Create Bag-of-Words features. (สร้าง features แบบ Bag-of-Words)
4. Train Logistic Regression with an 80/20 split. (เทรนโมเดล Logistic Regression ด้วยการแบ่ง train/test 80/20)
5. Report accuracy and the confusion matrix. (รายงานค่า accuracy และ confusion matrix)

In [1]:
import json
import re
from pathlib import Path
from urllib.request import urlretrieve

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

DATASET_URL = 'https://lief-assets-storage.sgp1.cdn.digitaloceanspaces.com/Test/dataset.txt'
DATASET_PATH = Path('dataset.txt')
STOPWORDS = {'is', 'the', 'and', 'a', 'an', 'of'}
RANDOM_STATE = 42

## 1. Load the dataset (โหลดชุดข้อมูล)

The first run downloads the supplied JSON dataset into this folder; later runs reuse it. (รันครั้งแรกจะดาวน์โหลดชุดข้อมูล JSON มาไว้ในโฟลเดอร์นี้ รอบถัดไปจะใช้ไฟล์ที่มีอยู่)

In [2]:
if not DATASET_PATH.exists():
    urlretrieve(DATASET_URL, DATASET_PATH)

records = json.loads(DATASET_PATH.read_text(encoding='utf-8'))
print(f'Dataset size: {len(records)}')
print(records[0])

Dataset size: 1000
{'text': 'I absolutely loved this movie, it was fantastic!', 'label': 1}


## 2. Preprocess text

The function lowercases text, removes punctuation, splits it into tokens, removes only the stopwords specified in the question, and joins the remaining tokens.

In [3]:
def preprocess(text: str) -> str:
    # แปลงเป็นพิมพ์เล็ก เพื่อให้คำเหมือนกันถูกนับเป็นคำเดียวกัน
    text = text.lower()
    # ลบเครื่องหมายวรรคตอน โดยแทนที่ด้วยช่องว่าง
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    # ตัดข้อความออกเป็นคำ (tokenize แบบง่ายด้วย split)
    tokens = text.split()
    # กรอง stopwords ที่กำหนดออก แล้วเชื่อมคำกลับเป็นข้อความ
    return ' '.join(token for token in tokens if token not in STOPWORDS)

# สร้างรายการข้อความที่ผ่าน preprocessing และ label สำหรับทุก record
texts = [preprocess(record['text']) for record in records]
labels = [int(record['label']) for record in records]
print('Before:', records[0]['text'])
print('After: ', texts[0])

Before: I absolutely loved this movie, it was fantastic!
After:  i absolutely loved this movie it was fantastic


## 3. Train and evaluate (เทรนโมเดลและประเมินผล)

`stratify=labels` keeps the positive/negative class ratio consistent across the 80% training and 20% testing sets. (stratify ช่วยให้สัดส่วน positive/negative เท่ากันในทั้งชุด train และ test)
We use a `CountVectorizer` to build a Bag-of-Words representation of the preprocessed text. (เราใช้ CountVectorizer แปลงข้อความที่ผ่าน preprocessing เป็น Bag-of-Words vector)

In [4]:
x_train, x_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=RANDOM_STATE, stratify=labels
)

model = Pipeline([
    ('bow', CountVectorizer()),
    ('logistic_regression', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
model.fit(x_train, y_train)
predictions = model.predict(x_test)

accuracy = accuracy_score(y_test, predictions)
matrix = confusion_matrix(y_test, predictions, labels=[0, 1])
print(f'Train / test: {len(x_train)} / {len(x_test)}')
print(f'Accuracy: {accuracy:.4f}')
print('Confusion matrix (rows=actual, columns=predicted; labels=[0, 1]):')
print(matrix)

Train / test: 800 / 200
Accuracy: 1.0000
Confusion matrix (rows=actual, columns=predicted; labels=[0, 1]):
[[100   0]
 [  0 100]]


## Result interpretation (ตีความผลลัพธ์)

In the confusion matrix, row 0/1 is the actual negative/positive label and column 0/1 is the predicted label. (ใน confusion matrix แถว 0/1 คือ label จริง negative/positive ส่วนคอลัมน์ 0/1 คือ label ที่โมเดลทำนาย)
A high score here should be interpreted with care because the supplied dataset contains repeated examples. (คะแนนสูงควรตีความระวัง เพราะชุดข้อมูลที่ให้มามีตัวอย่างซ้ำกันจำนวนมาก)